# Análisis YAPE: Contratos vs Fuente

Compara el sheet `CONTRATOS YAPE` contra la planilla fuente `Anexar1` para identificar:
- Registros faltantes en contratos (nuevos de abr/may 2026)
- Discrepancias en columnas automáticas
- Validación de la llave correcta (`CODIGO + CI`)

**Lee CLAUDE.md antes de modificar este notebook.**

In [3]:
import pandas as pd
import numpy as np

# ── Rutas de archivos ──
# Ajusta estas rutas si mueves los archivos
PATH_FUENTE  = 'PERSONAL YAPE.csv'
PATH_CONTRATOS = 'Copia de CONTROL DE CONTRATOS POR PROYECTOS.xlsx'
HOJA_CONTRATOS = 'CONTRATOS YAPE'

# Cargar fuente YAPE
df_yape = pd.read_csv(PATH_FUENTE, dtype=str, encoding='utf-8')
df_yape = df_yape.fillna('')
df_yape.columns = df_yape.columns.str.strip()

# Cargar sheet de contratos
df_contratos = pd.read_excel(PATH_CONTRATOS, sheet_name=HOJA_CONTRATOS, dtype=str)
df_contratos = df_contratos.fillna('')
df_contratos.columns = df_contratos.columns.str.strip()

print(f'Fuente YAPE:      {len(df_yape):>5} filas | {df_yape.shape[1]} columnas')
print(f'Contratos YAPE:   {len(df_contratos):>5} filas | {df_contratos.shape[1]} columnas')

Fuente YAPE:        990 filas | 23 columnas
Contratos YAPE:     976 filas | 21 columnas


## 1. Validación de llaves

In [4]:
# ── Llave CODIGO (sola) ──
print('=== LLAVE: solo CODIGO ===')
print(f'Duplicados en YAPE fuente:    {df_yape["CODIGO"].str.strip().duplicated().sum()}')
print(f'Duplicados en CONTRATOS:      {df_contratos["CODIGO"].str.strip().duplicated().sum()}')
print(f'Únicos en YAPE:               {df_yape["CODIGO"].str.strip().nunique()}')
print(f'Únicos en CONTRATOS:          {df_contratos["CODIGO"].str.strip().nunique()}')

print()

# ── Llave CODIGO + CI ──
df_yape['KEY']      = df_yape['CODIGO'].str.strip() + '|' + df_yape['C.I.'].str.strip()
df_contratos['KEY'] = df_contratos['CODIGO'].str.strip() + '|' + df_contratos['C.I.'].str.strip()

print('=== LLAVE: CODIGO + C.I. ===')
print(f'Duplicados en YAPE fuente:    {df_yape["KEY"].duplicated().sum()}')
print(f'Duplicados en CONTRATOS:      {df_contratos["KEY"].duplicated().sum()}')
print(f'Únicos en YAPE:               {df_yape["KEY"].nunique()}')
print(f'Únicos en CONTRATOS:          {df_contratos["KEY"].nunique()}')

=== LLAVE: solo CODIGO ===
Duplicados en YAPE fuente:    529
Duplicados en CONTRATOS:      516
Únicos en YAPE:               461
Únicos en CONTRATOS:          460

=== LLAVE: CODIGO + C.I. ===
Duplicados en YAPE fuente:    0
Duplicados en CONTRATOS:      147
Únicos en YAPE:               990
Únicos en CONTRATOS:          829


## 2. Registros faltantes en Contratos

In [5]:
keys_yape      = set(df_yape['KEY'])
keys_contratos = set(df_contratos['KEY'])

solo_en_yape      = keys_yape - keys_contratos
solo_en_contratos = keys_contratos - keys_yape
en_ambos          = keys_yape & keys_contratos

print(f'Match (en ambos):                   {len(en_ambos)}')
print(f'Solo en fuente YAPE (faltantes):    {len(solo_en_yape)}')
print(f'Solo en CONTRATOS (sin match):      {len(solo_en_contratos)}')

Match (en ambos):                   829
Solo en fuente YAPE (faltantes):    161
Solo en CONTRATOS (sin match):      0


In [6]:
# Detalle de faltantes en contratos
faltantes = df_yape[df_yape['KEY'].isin(solo_en_yape)].copy()

cols_vista = ['CODIGO', 'C.I.', 'NOMBRE', 'ESTADO (ACTIVO/INACTIVO)', 'FECHA DE INGRESO', 'CIUDAD']

print('=== FALTANTES EN CONTRATOS ===')
print(f'Total: {len(faltantes)}')
print()
print('Por estado:')
print(faltantes['ESTADO (ACTIVO/INACTIVO)'].value_counts())
print()

# Activos faltantes — estos SÍ deberían estar en contratos
activos_faltantes = faltantes[faltantes['ESTADO (ACTIVO/INACTIVO)'].str.upper().str.strip() == 'ACTIVO']
print(f'ACTIVOS faltantes (deben insertarse): {len(activos_faltantes)}')
activos_faltantes[cols_vista].sort_values('FECHA DE INGRESO')

=== FALTANTES EN CONTRATOS ===
Total: 161

Por estado:
ESTADO (ACTIVO/INACTIVO)
INACTIVO    114
ACTIVO       47
Name: count, dtype: int64

ACTIVOS faltantes (deben insertarse): 47


,CODIGO,C.I.,NOMBRE,ESTADO (ACTIVO/INACTIVO),FECHA DE INGRESO,CIUDAD
463,302,14384709,CRISTIAN TOLA ROSALES,ACTIVO,1/2/2026,COCHABAMBA
608,275,5977858,BORIS RAMIREZ PEREZ,ACTIVO,10/5/2024,LA PAZ
566,236,13843708,PAOLA HILAQUITA SIÑANI,ACTIVO,11/1/2024,LA PAZ
775,106,11400186,BERENISE MERUBIA CARDOZO,ACTIVO,11/1/2025,SANTA CRUZ
520,288,13562898,YESSICA SILES VELAZCO,ACTIVO,12/1/2025,COCHABAMBA
353,292,8792592,RODRIGO TERRAZAS FLORES,ACTIVO,12/1/2025,COCHABAMBA
523,308,9427968,FREDDY VEIZAGA VERDUGUEZ,ACTIVO,2/4/2026,COCHABAMBA
468,150,5396506,RENE OSINAGA ALVAREZ,ACTIVO,3/1/2026,SANTA CRUZ
375,157,5360229,NERIDA SUPEPI CHUVE,ACTIVO,3/17/2026,SANTA CRUZ
470,158,3370981,MARIA ISABEL MONTALBA MEDINA,ACTIVO,3/17/2026,SANTA CRUZ


In [7]:
# Filtrar específicamente abril y mayo 2026
def parse_fecha(f):
    try:
        return pd.to_datetime(f, format='%m/%d/%Y')
    except:
        try:
            return pd.to_datetime(f)
        except:
            return pd.NaT

faltantes = faltantes.copy()
faltantes['FECHA_PARSED'] = faltantes['FECHA DE INGRESO'].apply(parse_fecha)

abr_may_2026 = faltantes[
    (faltantes['FECHA_PARSED'].dt.year == 2026) &
    (faltantes['FECHA_PARSED'].dt.month.isin([4, 5]))
]

print(f'Faltantes con ingreso ABR/MAY 2026: {len(abr_may_2026)}')
print()
abr_may_2026[cols_vista + ['FECHA_PARSED']].sort_values('FECHA_PARSED')

Faltantes con ingreso ABR/MAY 2026: 28



,CODIGO,C.I.,NOMBRE,ESTADO (ACTIVO/INACTIVO),FECHA DE INGRESO,CIUDAD,FECHA_PARSED
550,163,8905079,ALEJANDRO SALVATIERRA VILLARROEL,ACTIVO,4/1/2026,SANTA CRUZ,2026-04-01
552,182,13398856,LUIS FERNANDO PACOSILLO CONDORI,ACTIVO,4/1/2026,SANTA CRUZ,2026-04-01
358,328,7705610,CBBA PROYECTO GESTORES,ACTIVO,4/6/2026,COCHABAMBA,2026-04-06
473,168,7705610,SCZ PROYECTO GESTORES,ACTIVO,4/6/2026,SANTA CRUZ,2026-04-06
471,418,7705610,LPZ PROYECTO GESTORES,ACTIVO,4/6/2026,LA PAZ,2026-04-06
382,175,3897209,DEYSI ALCON,ACTIVO,4/6/2026,SANTA CRUZ,2026-04-06
379,172,9760767,SEBASTIAN MARTINEZ ROCA,ACTIVO,4/6/2026,SANTA CRUZ,2026-04-06
453,176,7824950,JESUS CONRADO CAMACHO LIZARAZU,INACTIVO,4/7/2026,SANTA CRUZ,2026-04-07
454,177,9714935,MIGUEL CONDORI VALERIA,INACTIVO,4/7/2026,SANTA CRUZ,2026-04-07
551,179,15254135,DANNA MICHELLE VARGAS AVALOS,ACTIVO,4/9/2026,SANTA CRUZ,2026-04-09


## 3. Discrepancias en registros que sí existen en ambos

In [8]:
# Comparar columnas automáticas en registros que existen en ambos
# Mapeo fuente → contratos
MAPEO = {
    'CELULAR':                   'CELULAR',
    'ESTADO (ACTIVO/INACTIVO)':  'ESTADO',
    'CARGO':                     'CARGO',
    'SUPERVISOR':                'SUPERVISOR',
    'CIUDAD':                    'CIUDAD',
}

df_y_match = df_yape[df_yape['KEY'].isin(en_ambos)].set_index('KEY')
df_c_match = df_contratos[df_contratos['KEY'].isin(en_ambos)].set_index('KEY')

# Solo primer match por key en contratos (puede haber duplicados)
df_c_match = df_c_match[~df_c_match.index.duplicated(keep='first')]

discrepancias = []

for col_fuente, col_destino in MAPEO.items():
    if col_fuente not in df_y_match.columns or col_destino not in df_c_match.columns:
        continue
    
    comparacion = pd.DataFrame({
        'fuente':   df_y_match[col_fuente].str.strip().str.upper(),
        'contrato': df_c_match[col_destino].str.strip().str.upper()
    }).dropna()
    
    diff = comparacion[comparacion['fuente'] != comparacion['contrato']]
    discrepancias.append((col_fuente, len(diff)))
    
    if len(diff) > 0:
        print(f'\n⚠ {col_fuente} → {col_destino}: {len(diff)} discrepancias')
        print(diff.head(10))

print('\n=== RESUMEN DISCREPANCIAS ===')
for col, n in discrepancias:
    estado = '✓ OK' if n == 0 else f'⚠ {n} diferencias'
    print(f'{col:<35} {estado}')


⚠ ESTADO (ACTIVO/INACTIVO) → ESTADO: 16 discrepancias
                fuente contrato
KEY                            
111|9588758   INACTIVO   ACTIVO
164|4668762   INACTIVO   ACTIVO
165|9270089   INACTIVO   ACTIVO
166|9714936   INACTIVO   ACTIVO
167|15567682  INACTIVO   ACTIVO
247|3326009   INACTIVO   ACTIVO
268|7958673   INACTIVO   ACTIVO
284|6799045   INACTIVO   ACTIVO
290|6521548   INACTIVO   ACTIVO
296|9442119   INACTIVO   ACTIVO

⚠ SUPERVISOR → SUPERVISOR: 474 discrepancias
                                           fuente          contrato
KEY                                                                
100|5981781                    LIONEL CHABUR LOZA     LIONEL CHABUR
100|7992451          JOSE ALBERTO URQUIDI TORRICO      JOSE URQUIDI
101|7450894          JOSE ALBERTO URQUIDI TORRICO      JOSE URQUIDI
101|8278526                    LIONEL CHABUR LOZA     LIONEL CHABUR
102|12622032                   LIONEL CHABUR LOZA     LIONEL CHABUR
103|2442462             NAYELI WARA RAM

## 4. Duplicados en CONTRATOS YAPE

In [9]:
# Ver duplicados actuales en el sheet de contratos
dups_contratos = df_contratos[df_contratos['KEY'].duplicated(keep=False)].copy()
dups_contratos = dups_contratos.sort_values(['CODIGO', 'FECHA DE INGRESO'])

print(f'Filas duplicadas (mismo CODIGO+CI) en CONTRATOS: {len(dups_contratos)}')
print()

cols_dup = ['CODIGO', 'C.I.', 'NOMBRE COMPLETO', 'ESTADO', 'FECHA DE INGRESO']
dups_contratos[cols_dup].head(20)

Filas duplicadas (mismo CODIGO+CI) en CONTRATOS: 293



,CODIGO,C.I.,NOMBRE COMPLETO,ESTADO,FECHA DE INGRESO
358,10,8145005,Liliana Fernandez Rojas,INACTIVO,01/07/2025
86,10,8145005,Liliana Fernandez Rojas,INACTIVO,2025-07-01 00:00:00
846,102,5386582,Telma Banessa Saavedra Cayoja,INACTIVO,01/11/2025
27,102,5386582,Telma Banessa Saavedra Cayoja,INACTIVO,2025-11-01 00:00:00
847,105,3507313,Eliana Nayda Paz Mamani,INACTIVO,17/11/2025
655,105,3507313,Eliana Nayda Paz Mamani,INACTIVO,2025-11-17 00:00:00
910,106,12637660,Herlan Rodrigo Quispe Callisaya,INACTIVO,02/10/2024
242,106,12637660,Herlan Rodrigo Quispe Callisaya,INACTIVO,2024-10-02 00:00:00
851,111,9588758,Juan Pablo Flores Huallpa,INACTIVO,01/12/2025
225,111,9588758,Juan Pablo Flores Huallpa,ACTIVO,2025-12-01 00:00:00


## 5. Columnas manuales — validar que no se perdieron

In [10]:
# Columnas manuales: M=FECHA DE CONTRATO, P=FIRMA, Q=ESTADO DE CONTRATO, R=OBS, S=OBS SUPERVISOR, T=OBS SUBGERENTE
cols_manuales = ['FECHA DE CONTRATO', 'FIRMA', 'ESTADO DE CONTRATO', 'OBS', 'OBS SUPERVISOR', 'OBS SUBGERENTE']

print('=== COLUMNAS MANUALES: datos no vacíos ===')
for col in cols_manuales:
    if col in df_contratos.columns:
        no_vacios = (df_contratos[col].str.strip() != '').sum()
        print(f'{col:<25}: {no_vacios:>4} registros con dato')
    else:
        print(f'{col:<25}: columna NO encontrada')

=== COLUMNAS MANUALES: datos no vacíos ===
FECHA DE CONTRATO        :    7 registros con dato
FIRMA                    :    0 registros con dato
ESTADO DE CONTRATO       :  246 registros con dato
OBS                      :   22 registros con dato
OBS SUPERVISOR           :    2 registros con dato
OBS SUBGERENTE           :    0 registros con dato


## 6. Generar reporte de registros a insertar

In [11]:
# Lista limpia de ACTIVOS que faltan en contratos → estos son los que deben insertarse
activos_faltantes_clean = faltantes[
    faltantes['ESTADO (ACTIVO/INACTIVO)'].str.upper().str.strip() == 'ACTIVO'
][[
    'EMPRESA', 'CODIGO', 'C.I.', 'CELULAR', 'NOMBRE',
    'APELLIDO PATERNO', 'APELLIDO MATERNO', 'NOMBRES',
    'CARGO', 'CIUDAD', 'SUPERVISOR', 'FECHA DE INGRESO',
    'ESTADO (ACTIVO/INACTIVO)'
]].copy()

activos_faltantes_clean['FECHA_PARSED'] = activos_faltantes_clean['FECHA DE INGRESO'].apply(parse_fecha)
activos_faltantes_clean = activos_faltantes_clean.sort_values('FECHA_PARSED')

print(f'Total ACTIVOS a insertar: {len(activos_faltantes_clean)}')
activos_faltantes_clean.drop(columns='FECHA_PARSED')

Total ACTIVOS a insertar: 47


,EMPRESA,CODIGO,C.I.,CELULAR,NOMBRE,APELLIDO PATERNO,APELLIDO MATERNO,NOMBRES,CARGO,CIUDAD,SUPERVISOR,FECHA DE INGRESO,ESTADO (ACTIVO/INACTIVO)
574,YAPE,218,4468744,72215525,JOSE ALBERTO URQUIDI TORRICO,URQUIDI,TORRICO,JOSE ALBERTO,LIDER,COCHABAMBA,JOSE ALBERTO URQUIDI TORRICO,7/22/2024,ACTIVO
606,YAPE,273,6815939,65607126,MAYRA GURSEL ZALLES MORALES,ZALLES,MORALES,MAYRA GURSEL,AFILIADOR,LA PAZ,NAYELI WARA RAMOS LLANQUE,9/3/2024,ACTIVO
565,YAPE,235,6790254,69714550,NAYELI WARA RAMOS LLANQUE,RAMOS,LLANQUE,NAYELI WARA,LIDER,LA PAZ,NAYELI WARA RAMOS LLANQUE,9/3/2024,ACTIVO
608,YAPE,275,5977858,69194361,BORIS RAMIREZ PEREZ,RAMIREZ,PEREZ,BORIS,AFILIADOR,LA PAZ,NAYELI WARA RAMOS LLANQUE,10/5/2024,ACTIVO
566,YAPE,236,13843708,69745433,PAOLA HILAQUITA SIÑANI,HILAQUITA,SIÑANI,PAOLA,LIDER,LA PAZ,PAOLA HILAQUITA SIÑANI,11/1/2024,ACTIVO
623,YAPE,294,7087093,78847176,CINTHIA VERONICA MAMANI MAMANI,MAMANI,MAMANI,CINTHIA VERONICA,AFILIADOR,LA PAZ,ROSA CONDORI ALARCON,4/1/2025,ACTIVO
562,YAPE,232,10576388,75844371,LEYMER MARIO FERNANDEZ POMA,FERNANDEZ,POMA,LEYMER MARIO,SUPERVISOR,LA PAZ,BEX,5/1/2025,ACTIVO
576,YAPE,231,7412893,70360925,LEIDY CLARA GUTIERREZ FLORES,GUTIERREZ,FLORES,LEIDY CLARA,LIDER,COCHABAMBA,LEIDY CLARA GUTIERREZ FLORES,7/1/2025,ACTIVO
680,YAPE,219,14445306,61238666,HECTOR ADONAI ZAPATA VELARDE,ZAPATA,VELARDE,HECTOR ADONAI,AFILIADOR,COCHABAMBA,JOSE ALBERTO URQUIDI TORRICO,8/1/2025,ACTIVO
568,YAPE,238,7968944,69793053,SILVIA EUGENIA AQUINO FLORES,AQUINO,FLORES,SILVIA EUGENIA,LIDER,LA PAZ,SILVIA EUGENIA AQUINO FLORES,8/1/2025,ACTIVO


In [12]:
# Guardar reporte a CSV para revisión
activos_faltantes_clean.drop(columns='FECHA_PARSED').to_csv('reporte_faltantes_yape.csv', index=False, encoding='utf-8-sig')
print('Guardado: reporte_faltantes_yape.csv')

Guardado: reporte_faltantes_yape.csv


## 7. Código Apps Script corregido — llave CODIGO + CI

El siguiente snippet reemplaza la construcción del mapa en `actualizarYAPE()`.

```javascript
// ── CAMBIO CLAVE: llave = CODIGO + "|" + CI (no celular) ──

// Al leer destino:
datosDestino.forEach(function(fila, index) {
  var codigo = String(fila[1] || '').trim();  // col B
  var ci     = String(fila[2] || '').trim();  // col C  ← NUEVO
  var clave  = codigo + '|' + ci;             // ← llave compuesta
  
  if (codigo !== '' && ci !== '') {
    codigosDestino.add(clave);
    mapaDestino[clave] = { filaReal: index + 2, datos: fila };
  }
});

// Al procesar origen:
datosOrigen.forEach(function(filaOrigen) {
  var codigo = String(filaOrigen[MAPEO_YAPE.codigo] || '').trim();
  var ci     = String(filaOrigen[MAPEO_YAPE.ci]     || '').trim();  // ← NUEVO
  var clave  = codigo + '|' + ci;                                   // ← llave compuesta
  
  // ... resto de la lógica usando `clave` en vez de la combinación anterior
});
```

Con esta llave, cada persona que rotó por el mismo CODIGO queda como fila separada, que es el comportamiento correcto dado que el sheet de contratos ya los tiene así.